# Notebook 04: Fabrication Analysis — Source PDF Pipeline

Runs the LangGraph RAG extraction pipeline on every patient case where the original
AI model produced a fabrication (label=3), using the **original source PDFs** from
the Breast Bot OneDrive folder.

## Workflow
```
ai_fabrications_dataset.xlsx (14 patients, all with status_ai=3)
  ↓ normalize surgeon last name + patient_initials
  ↓ join to case_id_mapping.csv → get all source PDF original_paths
  ↓ OCR each PDF (pytesseract + fitz, parallel, with caching)
  ↓ concatenate PDFs per patient
  ↓ identify which features were fabricated per patient
  ↓ LangGraph RAG pipeline (feature_queue = fabricated features only)
  ↓ compare pipeline verdicts vs original fabrication labels
  ↓ figures + auditable JSON/CSV output
```

**Input data (private, not in repo):**
- `C:\Users\jamesr4\loc\data_private\raw\ai_fabrications_dataset.xlsx`
- `C:\Users\jamesr4\loc\data_private\breast_bot_deidentified\case_id_mapping.csv`
- Source PDFs at: `C:\Users\jamesr4\OneDrive - Memorial Sloan Kettering Cancer Center\Moo, Tracy-Ann's files - Breast Bot Project\{surgeon}\{case_folder}\`

## 0. Environment & Imports

In [ ]:
import os, sys, json, re, time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import fitz
import pytesseract
from PIL import Image
from dotenv import load_dotenv
from tqdm.notebook import tqdm

load_dotenv()

PROJECT_ROOT   = Path(os.getenv("PROJECT_ROOT",
    r"C:\Users\jamesr4\OneDrive - Memorial Sloan Kettering Cancer Center"
    r"\Documents\GitHub\llm_summarization_br_ca"))
DATA_PRIVATE   = Path(os.getenv("DATA_PRIVATE_DIR", r"C:\Users\jamesr4\loc\data_private"))
ONEDRIVE_ROOT  = Path(r"C:\Users\jamesr4\OneDrive - Memorial Sloan Kettering Cancer Center"
                      r"\Moo, Tracy-Ann's files - Breast Bot Project")

FAB_XLSX       = DATA_PRIVATE / "raw" / "ai_fabrications_dataset.xlsx"
MAPPING_CSV    = DATA_PRIVATE / "breast_bot_deidentified" / "case_id_mapping.csv"
OCR_CACHE_DIR  = DATA_PRIVATE / "ocr_cache_source"
RUN_OUT_DIR    = PROJECT_ROOT / "experiments" / "runs" / "fab_source_pdf"
REPORTS_DIR    = PROJECT_ROOT / "reports"

for d in [OCR_CACHE_DIR, RUN_OUT_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT))

pytesseract.pytesseract.tesseract_cmd = (
    r"C:\Users\jamesr4\AppData\Local\miniforge3\Library\bin\tesseract.exe"
)
os.environ["TESSDATA_PREFIX"] = (
    r"C:\Users\jamesr4\AppData\Local\miniforge3\share\tessdata"
)

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)

print(f"PROJECT_ROOT  : {PROJECT_ROOT}")
print(f"FAB_XLSX      : {FAB_XLSX}")
print(f"ONEDRIVE_ROOT : {ONEDRIVE_ROOT}")
print(f"OCR cache     : {OCR_CACHE_DIR}")
print(f"API key set   : {'ANTHROPIC_API_KEY' in os.environ}")

## 1. Load Fabrications Dataset & Map to Source PDFs

In [ ]:
df_fab = pd.read_excel(FAB_XLSX)
mapping = pd.read_csv(MAPPING_CSV)

print(f"Fabrications dataset : {df_fab.shape}")
print(f"Case ID mapping      : {mapping.shape}")

AI_COLS = [c for c in df_fab.columns if c.endswith("_status_ai")]
for c in AI_COLS:
    df_fab[c] = pd.to_numeric(df_fab[c], errors="coerce")

fab_mask = (df_fab[AI_COLS] == 3).any(axis=1)
print(f"\nRows with at least one AI fabrication (status=3): {fab_mask.sum()} / {len(df_fab)}")

print("\nFabrication counts by feature:")
feat_fabs = pd.Series(
    {c.replace("_status_ai", ""): (df_fab[c] == 3).sum() for c in AI_COLS}
).sort_values(ascending=False)
print(feat_fabs[feat_fabs > 0].to_string())

In [ ]:
_SURGEON_MAP = {
    "el tamer": "el tamer", "el-tamer": "el tamer",
    "sacchini": "sacchini",
    "giannakou": "giannakou",
    "montagna": "montag", "montag": "montag",
    "lisa allen": "allen",
}

def _norm_surgeon(s: str) -> str:
    sl = str(s).strip().lower()
    return _SURGEON_MAP.get(sl, sl)

df_fab["surgeon_last"] = df_fab["surgeon"].str.split(",").str[0].str.strip()
df_fab["surgeon_norm"] = df_fab["surgeon_last"].apply(_norm_surgeon)

mapping["surgeon_norm"]    = mapping["surgeon"].apply(_norm_surgeon)
mapping["folder_initials"] = mapping["case_folder"].str.split("_").str[1].str.upper()

df_merged = df_fab.merge(
    mapping[["case_id", "surgeon_norm", "folder_initials", "case_folder",
             "original_filename", "original_path", "deidentified_path", "status"]],
    left_on=["surgeon_norm", "patient_initials"],
    right_on=["surgeon_norm", "folder_initials"],
    how="left",
)

print(f"Merged rows          : {len(df_merged)}")
print(f"Unmatched (no case_id): {df_merged['case_id'].isna().sum()}")
print(f"Unique patients       : {df_merged['mrn'].nunique()}")
print(f"Unique case_ids       : {df_merged['case_id'].nunique()}")

# Build per-patient PDF list: mrn -> list of original_path (OneDrive source PDFs)
patient_pdfs = {}
patient_meta = {}
for mrn, grp in df_merged.dropna(subset=["original_path"]).groupby("mrn"):
    pdfs = [Path(p) for p in grp["original_path"].unique()]
    pdfs = [p for p in pdfs if p.exists()]
    patient_pdfs[int(mrn)] = sorted(pdfs)
    row0 = grp.iloc[0]
    patient_meta[int(mrn)] = {
        "mrn": int(mrn),
        "surgeon": row0["surgeon"],
        "surgeon_last": row0["surgeon_last"],
        "patient_initials": row0["patient_initials"],
        "case_folder": row0["case_folder"],
    }

pdf_counts = {mrn: len(v) for mrn, v in patient_pdfs.items()}
print(f"\nPatients with matched PDFs : {len(patient_pdfs)}")
print(f"Total source PDFs          : {sum(pdf_counts.values())}")
if pdf_counts:
    print(f"PDFs/patient range         : {min(pdf_counts.values())}–{max(pdf_counts.values())}")

## 2. OCR Source PDFs (with caching)

In [ ]:
OCR_DPI     = 200
OCR_WORKERS = 4
OCR_PSM     = "--psm 6"

def ocr_pdf(pdf_path: Path, dpi: int = OCR_DPI) -> str:
    """OCR one PDF → full text string. Returns cached version if available."""
    cache_file = OCR_CACHE_DIR / (pdf_path.stem + ".txt")
    if cache_file.exists():
        return cache_file.read_text(encoding="utf-8")
    try:
        doc   = fitz.open(str(pdf_path))
        pages = []
        zoom  = dpi / 72.0
        mat   = fitz.Matrix(zoom, zoom)
        for i in range(doc.page_count):
            pix  = doc.load_page(i).get_pixmap(matrix=mat, alpha=False)
            img  = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
            text = pytesseract.image_to_string(img, config=OCR_PSM)
            pages.append(f"[PAGE {i+1}]\n{text}")
        doc.close()
        full = "\n\n".join(pages)
        cache_file.write_text(full, encoding="utf-8")
        return full
    except Exception as e:
        return f"[OCR_ERROR: {pdf_path.name}: {e}]"


def ocr_all_pdfs(pdf_list: list, workers: int = OCR_WORKERS) -> dict:
    """OCR a list of PDFs in parallel. Returns {pdf_path: text}."""
    results = {}
    cached  = sum(1 for p in pdf_list if (OCR_CACHE_DIR / (p.stem + ".txt")).exists())
    print(f"  {cached}/{len(pdf_list)} PDFs already cached")
    with ThreadPoolExecutor(max_workers=workers) as ex:
        futures = {ex.submit(ocr_pdf, p): p for p in pdf_list}
        for fut in tqdm(as_completed(futures), total=len(futures), desc="OCR"):
            pdf_path = futures[fut]
            try:
                results[pdf_path] = fut.result()
            except Exception as e:
                results[pdf_path] = f"[FUTURE_ERROR: {e}]"
    return results


all_pdfs = sorted({p for pdfs in patient_pdfs.values() for p in pdfs})
print(f"Unique source PDFs to OCR: {len(all_pdfs)}")
print("Starting OCR (cached results are instant)...\n")

t0 = time.time()
ocr_results = ocr_all_pdfs(all_pdfs)
elapsed = time.time() - t0

print(f"\nOCR complete in {elapsed/60:.1f} min")
char_lens = [len(v) for v in ocr_results.values() if not str(v).startswith("[")]
if char_lens:
    print(f"Chars extracted — median: {int(np.median(char_lens))}, max: {max(char_lens)}")

## 3. Build Per-Patient Concatenated OCR Documents

In [ ]:
_DOC_ORDER = [
    "mammo", "mri", "ultrasound", "us", "imaging",
    "path", "pathology", "receptor", "genetic",
]

def _doc_sort_key(p: Path) -> int:
    name = p.stem.lower()
    for i, kw in enumerate(_DOC_ORDER):
        if kw in name:
            return i
    return len(_DOC_ORDER)


patient_ocr = {}
for mrn, pdf_paths in patient_pdfs.items():
    sorted_pdfs = sorted(pdf_paths, key=_doc_sort_key)
    sections = []
    for pdf in sorted_pdfs:
        text = ocr_results.get(pdf, "")
        if text and not str(text).startswith("[OCR_ERROR") and not str(text).startswith("[FUTURE_ERROR"):
            sections.append(f"=== SOURCE: {pdf.name} ===\n{text}")
    patient_ocr[mrn] = "\n\n".join(sections)

ok_patients = {mrn for mrn, txt in patient_ocr.items() if len(txt) > 200}
print(f"Patients with usable OCR text: {len(ok_patients)} / {len(patient_ocr)}")

for mrn in sorted(ok_patients):
    meta = patient_meta[mrn]
    print(f"  MRN {mrn:>10} | {meta['surgeon_last']:<15} | initials={meta['patient_initials']:<5} "
          f"| pdfs={len(patient_pdfs[mrn])} | chars={len(patient_ocr[mrn]):,}")

## 4. Identify Fabricated Features Per Patient

In [ ]:
COL_TO_FEATURE = {
    "lesion_size_status_ai":                        "feature_1_lesion_size",
    "laterality_status_ai":                         "feature_2_laterality",
    "lesion_location_status_ai":                    "feature_3_lesion_location",
    "calcifications_asymmetry_status_ai":           "feature_4_calcifications_asymmetry",
    "additional_enhancement_mri_status_ai":         "feature_5_additional_enhancement_mri",
    "extent_status_ai":                             "feature_6_extent",
    "accurate_clip_placement_status_ai":            "feature_7_accurate_clip_placement",
    "workup_recommendation_status_ai":              "feature_8_workup_recommendation",
    "Lymph node_status_ai":                         "feature_9_lymph_node",
    "chronology_preserved_status_ai":               "feature_10_chronology_preserved",
    "biopsy_method_status_ai":                      "feature_11_biopsy_method",
    "invasive_component_size_pathology_status_ai":  "feature_12_invasive_component_size_pathology",
    "histologic_diagnosis_status_ai":               "feature_13_histologic_diagnosis",
    "receptor_status_ai":                           "feature_14_receptor_status",
}

LABEL_DECODE = {1: "CORRECT", 2: "OMISSION", 3: "FABRICATION"}

patient_fab_features = {}  # mrn -> {feature_name: "FABRICATION"}
for _, row in df_fab.iterrows():
    mrn = int(row["mrn"])
    fab_feats = {}
    for col, feat in COL_TO_FEATURE.items():
        if col in row.index:
            val = row[col]
            if pd.notna(val) and int(val) == 3:
                fab_feats[feat] = "FABRICATION"
    patient_fab_features[mrn] = fab_feats

total_fab_features = sum(len(v) for v in patient_fab_features.values())
print(f"Patients               : {len(patient_fab_features)}")
print(f"Total fabricated features to re-run: {total_fab_features}")
print()

for mrn, feats in sorted(patient_fab_features.items()):
    meta = patient_meta.get(mrn, {})
    feat_names = [f.replace("feature_", "").replace("_", " ") for f in feats]
    print(f"  MRN {mrn:>10} | {meta.get('surgeon_last','?'):<15} {meta.get('patient_initials','?'):<5} "
          f"→ {feat_names}")

## 5. Run LangGraph RAG Pipeline on Fabrication Cases

Re-extracts only the fabricated features for each patient using the source PDF text.
Results are cached incrementally to `experiments/runs/fab_source_pdf/pipeline_results.json`.

In [ ]:
DRY_RUN  = False   # Set True to skip API calls and test mapping/OCR only
MODEL_ID  = "claude-3-5-sonnet-20241022"
PROMPT_ID = "rag_verify_v1"

RESULTS_CACHE = RUN_OUT_DIR / "pipeline_results.json"
if RESULTS_CACHE.exists():
    with open(RESULTS_CACHE) as f:
        all_results: dict = json.load(f)
    print(f"Loaded {len(all_results)} cached pipeline results from {RESULTS_CACHE}")
else:
    all_results = {}
    print("No cache found — will run from scratch")

print(f"DRY_RUN = {DRY_RUN}")

In [ ]:
from src.workflows.orchestration import run_single_case
from src.utils.io_utils import generate_run_id

run_id = generate_run_id()
print(f"Run ID: {run_id}")

patients_to_run = [
    mrn for mrn in ok_patients
    if str(mrn) not in all_results
]
print(f"Patients to run  : {len(patients_to_run)}")
print(f"Already cached   : {len(ok_patients) - len(patients_to_run)}")

failed_cases = []

for mrn in tqdm(patients_to_run, desc="Pipeline"):
    ocr_text   = patient_ocr[mrn]
    info       = patient_meta.get(mrn, {})
    error_feats = list(patient_fab_features.get(mrn, {}).keys())
    if not error_feats:
        continue

    case_id = f"MRN_{mrn}_{info.get('patient_initials', 'XX')}"

    if DRY_RUN:
        all_results[str(mrn)] = {
            "case_id": case_id, "mrn": mrn, "dry_run": True,
            "features": {f: {"value": "DRY_RUN", "verdict": None} for f in error_feats},
        }
        continue

    try:
        result = run_single_case(
            case_id=case_id,
            ocr_text=ocr_text,
            prompt_id=PROMPT_ID,
            model_id=MODEL_ID,
            feature_queue=error_feats,
            run_id=run_id,
        )
        result["mrn"] = mrn
        result["original_fab_features"] = patient_fab_features.get(mrn, {})
        result["ai_fab_comment"]    = df_fab.loc[df_fab["mrn"] == mrn, "ai_fab_comment"].values[0] \
                                       if "ai_fab_comment" in df_fab.columns else ""
        result["ai_fab_justification"] = df_fab.loc[df_fab["mrn"] == mrn, "ai_fab_justification"].values[0] \
                                          if "ai_fab_justification" in df_fab.columns else ""
        all_results[str(mrn)] = result

        with open(RESULTS_CACHE, "w") as f:
            json.dump(all_results, f, indent=2, default=str)

    except Exception as e:
        print(f"  ERROR MRN {mrn}: {e}")
        failed_cases.append({"mrn": mrn, "error": str(e)})

print(f"\nDone. Completed: {len(all_results)}  Failed: {len(failed_cases)}")

## 6. Build Comparison DataFrame

In [ ]:
rows = []
for mrn_str, result in all_results.items():
    mrn  = int(mrn_str)
    info = patient_meta.get(mrn, {})
    orig_fab = result.get("original_fab_features", patient_fab_features.get(mrn, {}))

    for feat, feat_data in result.get("features", {}).items():
        if feat not in orig_fab:
            continue  # only include fabricated features

        # Look up reviewer comment for this patient
        fab_row = df_fab[df_fab["mrn"] == mrn]
        ai_comment = fab_row["ai_fab_comment"].values[0] if len(fab_row) > 0 and "ai_fab_comment" in df_fab.columns else ""
        justification = fab_row["ai_fab_justification"].values[0] if len(fab_row) > 0 and "ai_fab_justification" in df_fab.columns else ""

        rows.append({
            "mrn":                     mrn,
            "patient_initials":        info.get("patient_initials"),
            "surgeon":                 info.get("surgeon_last"),
            "case_folder":             info.get("case_folder"),
            "feature_name":            feat,
            "original_ai_error":       "FABRICATION",
            "pipeline_value":          feat_data.get("value"),
            "pipeline_verdict":        feat_data.get("verdict"),
            "pipeline_confidence":     feat_data.get("confidence"),
            "pipeline_supported":      feat_data.get("supported"),
            "verification_confidence": feat_data.get("verification_confidence"),
            "verification_quote":      feat_data.get("verification_quote"),
            "verification_method":     feat_data.get("verification_method"),
            "retrieval_attempts":      feat_data.get("retrieval_attempts", 0),
            "evidence":                feat_data.get("evidence"),
            "ai_fab_comment":          ai_comment,
            "ai_fab_justification":    justification,
        })

df_cmp = pd.DataFrame(rows)
print(f"Comparison rows: {len(df_cmp)}")

if not df_cmp.empty:
    print("\nPipeline verdict distribution:")
    print(df_cmp["pipeline_verdict"].value_counts(dropna=False))

df_cmp.to_csv(RUN_OUT_DIR / "comparison_results.csv", index=False)
print(f"\nSaved: {RUN_OUT_DIR / 'comparison_results.csv'}")

## 7. Classify Recovery Status

In [ ]:
if df_cmp.empty:
    print("No comparison data yet. Run pipeline first.")
else:
    def classify_recovery(row):
        pipe = row["pipeline_verdict"]
        if pipe == "CORRECT":
            return "RECOVERED"
        elif pipe == "FABRICATION":
            return "CONFIRMED_FABRICATION"
        elif pipe == "UNCERTAIN":
            return "UNCERTAIN"
        elif pipe is None or (isinstance(pipe, float) and np.isnan(pipe)):
            return "NOT_RUN"
        else:
            return "OTHER_ERROR"

    df_cmp["recovery_status"] = df_cmp.apply(classify_recovery, axis=1)

    evaluated = df_cmp[df_cmp["recovery_status"] != "NOT_RUN"]
    total = len(evaluated)
    rc = df_cmp["recovery_status"].value_counts()

    print("=== Recovery Status (Fabrications Only) ===")
    print(rc.to_string())
    print()
    if total > 0:
        recovered  = rc.get("RECOVERED", 0)
        confirmed  = rc.get("CONFIRMED_FABRICATION", 0)
        uncertain  = rc.get("UNCERTAIN", 0)
        print(f"Recovery rate          : {recovered/total:.1%}  ({recovered}/{total})")
        print(f"Confirmed fabrications : {confirmed/total:.1%}  ({confirmed}/{total})")
        print(f"Uncertain              : {uncertain/total:.1%}  ({uncertain}/{total})")

## 8. Figure — Pipeline Recovery vs Confirmed Fabrications

In [ ]:
RC_COLORS = {
    "RECOVERED":             "#2ecc71",
    "CONFIRMED_FABRICATION": "#e74c3c",
    "UNCERTAIN":             "#95a5a6",
    "OTHER_ERROR":           "#9b59b6",
    "NOT_RUN":               "#ecf0f1",
}

if "recovery_status" not in df_cmp.columns or df_cmp.empty:
    print("Run cells 6–7 first.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # ── Left: stacked bar by feature ──────────────────────────────────────────
    feat_rc = (
        df_cmp.groupby(["feature_name", "recovery_status"])
        .size()
        .unstack(fill_value=0)
    )
    status_order = [s for s in RC_COLORS if s in feat_rc.columns]
    feat_rc = feat_rc[status_order]
    feat_rc.index = (
        feat_rc.index
        .str.replace(r"^feature_\d+_", "", regex=True)
        .str.replace("_", " ")
        .str.title()
    )
    feat_rc.plot(
        kind="barh", stacked=True,
        color=[RC_COLORS[s] for s in status_order],
        ax=axes[0], edgecolor="white", linewidth=1.2,
    )
    axes[0].set_title("Pipeline Response by Feature\n(AI fabrication cases only)",
                      fontweight="bold")
    axes[0].set_xlabel("Count")
    axes[0].legend(title="Pipeline Result", loc="lower right", fontsize=9)

    # ── Right: overall donut ───────────────────────────────────────────────────
    counts = df_cmp["recovery_status"].value_counts()
    wedge_colors = [RC_COLORS.get(k, "#bdc3c7") for k in counts.index]
    axes[1].pie(
        counts.values,
        labels=counts.index,
        colors=wedge_colors,
        autopct="%1.1f%%",
        startangle=90,
        pctdistance=0.8,
        wedgeprops={"linewidth": 2, "edgecolor": "white"},
    )
    axes[1].set_title("Overall Pipeline Recovery Rate", fontweight="bold")

    plt.suptitle(
        f"LangGraph RAG Pipeline vs Original AI Fabrications  (n={len(df_cmp)} feature-level)",
        fontsize=13, fontweight="bold",
    )
    plt.tight_layout()
    save_path = REPORTS_DIR / "fab_pipeline_recovery.png"
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {save_path}")

## 9. Figure — Verification Confidence for Fabrication Cases

In [ ]:
if df_cmp.empty or "verification_confidence" not in df_cmp.columns:
    print("No data.")
else:
    vc_data = df_cmp.dropna(subset=["verification_confidence"])

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Violin: verification confidence by recovery status
    if len(vc_data) > 1:
        palette = {k: v for k, v in RC_COLORS.items() if k in vc_data["recovery_status"].unique()}
        sns.violinplot(
            data=vc_data, x="recovery_status", y="verification_confidence",
            palette=palette, inner="box", cut=0, ax=axes[0],
        )
        axes[0].axhline(0.8, color="black", linestyle="--", alpha=0.5, label="Pass threshold")
        axes[0].set_title("Verification Confidence by Pipeline Result",
                          fontweight="bold")
        axes[0].set_ylabel("Verification Confidence")
        axes[0].set_xlabel("")
        axes[0].tick_params(axis="x", rotation=20)
        axes[0].legend()

    # Scatter: pipeline_confidence vs verification_confidence
    conf_data = df_cmp.dropna(subset=["pipeline_confidence", "verification_confidence"])
    if len(conf_data) > 0:
        for status, grp in conf_data.groupby("recovery_status"):
            color = RC_COLORS.get(status, "#bdc3c7")
            axes[1].scatter(
                grp["pipeline_confidence"], grp["verification_confidence"],
                label=status, color=color, alpha=0.75, s=70,
                edgecolors="white", linewidths=0.5,
            )
        axes[1].axhline(0.8,  color="red",  linestyle="--", alpha=0.4)
        axes[1].axvline(0.75, color="blue", linestyle="--", alpha=0.4)
        axes[1].set_xlabel("Pipeline Extraction Confidence")
        axes[1].set_ylabel("Verification Confidence")
        axes[1].set_title("Extraction vs Verification Confidence",
                          fontweight="bold")
        axes[1].legend(title="Pipeline Result", fontsize=8)

    plt.suptitle("Confidence Analysis — AI Fabrication Cases",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    save_path = REPORTS_DIR / "fab_verification_confidence.png"
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {save_path}")

## 10. Audit Table — Confirmed Fabrications with Reviewer Comments

In [ ]:
if df_cmp.empty:
    print("No data.")
else:
    display_cols = [
        "mrn", "patient_initials", "surgeon",
        "feature_name", "original_ai_error", "pipeline_verdict", "recovery_status",
        "pipeline_value", "pipeline_confidence",
        "verification_confidence", "verification_quote",
        "ai_fab_comment", "ai_fab_justification", "evidence",
    ]
    display_cols = [c for c in display_cols if c in df_cmp.columns]

    detail = df_cmp[display_cols].copy()
    detail["feature_name"] = (
        detail["feature_name"]
        .str.replace(r"^feature_\d+_", "", regex=True)
        .str.replace("_", " ")
        .str.title()
    )

    sort_order = {"RECOVERED": 0, "CONFIRMED_FABRICATION": 1, "UNCERTAIN": 2,
                  "OTHER_ERROR": 3, "NOT_RUN": 4}
    if "recovery_status" in detail.columns:
        detail["_sort"] = detail["recovery_status"].map(sort_order).fillna(5)
        detail = detail.sort_values(["_sort", "mrn", "feature_name"]).drop(columns=["_sort"])

    audit_path = RUN_OUT_DIR / "audit_table.csv"
    detail.to_csv(audit_path, index=False)
    print(f"Full audit table saved: {audit_path}")
    print(f"Rows: {len(detail)}")
    print()

    # Show confirmed fabrications
    if "recovery_status" in detail.columns:
        confirmed = detail[detail["recovery_status"] == "CONFIRMED_FABRICATION"]
        print(f"=== CONFIRMED FABRICATIONS (n={len(confirmed)}) ===")
        if len(confirmed) > 0:
            show_cols = [c for c in ["mrn", "feature_name", "pipeline_value",
                                     "pipeline_confidence", "verification_quote",
                                     "ai_fab_comment"] if c in confirmed.columns]
            print(confirmed[show_cols].to_string(index=False))

## 11. Save Final Summary Report

In [ ]:
from datetime import datetime

summary = {
    "run_timestamp": datetime.utcnow().isoformat(),
    "run_id":   run_id,
    "model_id": MODEL_ID,
    "prompt_id": PROMPT_ID,
    "dry_run":  DRY_RUN,
    "input": {
        "fabrication_cases": len(df_fab),
        "patients_with_matched_pdfs": len(patient_pdfs),
        "patients_with_ocr": len(ok_patients),
        "total_source_pdfs": sum(pdf_counts.values()),
        "total_fabricated_features": total_fab_features,
    },
    "pipeline": {
        "patients_processed": len(all_results),
        "patients_failed":    len(failed_cases),
        "feature_rows_in_comparison": len(df_cmp),
    },
}

if "recovery_status" in df_cmp.columns and not df_cmp.empty:
    rc = df_cmp["recovery_status"].value_counts()
    evaluated_n = len(df_cmp[df_cmp["recovery_status"] != "NOT_RUN"])
    summary["recovery"] = {
        "total_evaluated":     evaluated_n,
        "recovered":           int(rc.get("RECOVERED", 0)),
        "confirmed_fab":       int(rc.get("CONFIRMED_FABRICATION", 0)),
        "uncertain":           int(rc.get("UNCERTAIN", 0)),
        "recovery_rate":       round(rc.get("RECOVERED", 0) / evaluated_n, 3) if evaluated_n else None,
    }

summary_path = RUN_OUT_DIR / "run_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print("=" * 60)
print("RUN SUMMARY — Fabrication Source PDF Pipeline")
print("=" * 60)
print(json.dumps(summary, indent=2))
print(f"\nOutputs in: {RUN_OUT_DIR}")
print(f"Figures in: {REPORTS_DIR}")